##APACHE AIRFLOW - DAG

In [ ]:
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.standard.operators.bash import BashOperator
from airflow.providers.standard.operators.empty import EmptyOperator

default_args = {
    "owner": "data_engineering_lab",
    "depends_on_past": False,
    "start_date": datetime(2026, 1, 1),
    "email_on_failure": False,
    "email_on_retry": False,
    "retries": 1,
    "retry_delay": timedelta(minutes=5),
}

with DAG(
    "university_etl_orchestration",
    default_args=default_args,
    description="Lab assignment for API and Flat File ETL orchestration",
    schedule=timedelta(days=1),
    catchup=False,
) as dag:

    start_pipeline = EmptyOperator(task_id="start_pipeline")

    execute_extraction = BashOperator(
        task_id="run_extraction_script",
        bash_command="python3 /absolute/path/to/data_extraction.py",
    )

    pipeline_complete = BashOperator(
        task_id="log_pipeline_success",
        bash_command='echo "ETL Execution completed successfully at $(date)"',
    )

    start_pipeline >> execute_extraction >> pipeline_complete

In [ ]:
# We test the tasks sequentially inside the notebook context
# to bypass metadata database serialization constraints in Airflow 3
print("Executing 'start_pipeline'...")
start_pipeline.execute(context={})

print("\nExecuting 'run_extraction_script'...")
try:
    execute_extraction.execute(context={})
except Exception as e:
    print(f"Task execution failed as expected (since file doesn't exist): {e}")

print("\nExecuting 'log_pipeline_success'...")
pipeline_complete.execute(context={})

Executing 'start_pipeline'...
2026-10-03T20:09:30.195998Z [warning  ] EmptyOperator.execute cannot be called outside of the Task Runner! [airflow.task.operators.airflow.providers.standard.operators.empty.EmptyOperator] loc=operator.py:456

Executing 'run_extraction_script'...
2026-10-03T20:09:30.199116Z [warning  ] BashOperator.execute cannot be called outside of the Task Runner! [airflow.task.operators.airflow.providers.standard.operators.bash.BashOperator] loc=operator.py:456
2026-10-03T20:09:30.200475Z [info     ] Tmp dir root location: /tmp    [airflow.task.hooks.airflow.providers.standard.hooks.subprocess.SubprocessHook] loc=subprocess.py:78
2026-10-03T20:09:30.203185Z [info     ] Running command: ['/usr/bin/bash', '-c', 'python3 /absolute/path/to/data_extraction.py'] [airflow.task.hooks.airflow.providers.standard.hooks.subprocess.SubprocessHook] loc=subprocess.py:80
2026-10-03T20:09:30.205703Z [info     ] Output:                        [airflow.task.hooks.airflow.providers.standa

'ETL Execution completed successfully at Sat Oct  3 08:09:30 PM UTC 2026'